# Notebook 02 — Data Cleaning & ETL Pipeline

**Project:** Why Startups Fail — VC Investment Pattern Analysis  
**Team:** Section C, Group 17  

---
**Objective:** Build a fully documented, reproducible Python ETL pipeline that transforms the raw VC investment dataset into a clean, analysis-ready file. Every transformation is logged with a reason.

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# ETL log to document every transformation
etl_log = []

def log_step(step_num, description, before_shape, after_shape, detail=''):
    dropped_rows = before_shape[0] - after_shape[0]
    etl_log.append({
        'Step': step_num,
        'Description': description,
        'Rows Before': before_shape[0],
        'Rows After': after_shape[0],
        'Rows Dropped': dropped_rows,
        'Detail': detail
    })
    print(f'[Step {step_num}] {description} | Rows: {before_shape[0]:,} → {after_shape[0]:,} ({-dropped_rows:+,})')

print('ETL pipeline initialized.')

ETL pipeline initialized.


In [4]:
RAW_PATH = '/content/investments_VC (1).csv'


In [6]:
PROCESSED_PATH = '../data/processed/startups_cleaned.csv'

df = pd.read_csv(RAW_PATH, encoding='latin-1', low_memory=False)
initial_shape = df.shape
print(f'Raw dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

Raw dataset loaded: 54,294 rows x 39 columns


In [7]:
before = df.shape
df.columns = df.columns.str.strip()
log_step(1, 'Strip whitespace from column names', before, df.shape,
         'Columns like " market " and " funding_total_usd " had leading/trailing spaces')

[Step 1] Strip whitespace from column names | Rows: 54,294 → 54,294 (+0)


In [8]:
before = df.shape

def clean_funding(val):
    """Convert funding string like '1,750,000' or ' - ' to float."""
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().replace(',', '').replace(' ', '')
    if val_str in ['-', '', 'nan', 'None']:
        return np.nan
    try:
        return float(val_str)
    except:
        return np.nan

df['funding_total_usd'] = df['funding_total_usd'].apply(clean_funding)
log_step(2, 'Convert funding_total_usd to numeric', before, df.shape,
         'Stripped commas, dashes, spaces; converted to float')

[Step 2] Convert funding_total_usd to numeric | Rows: 54,294 → 54,294 (+0)


In [9]:
before = df.shape

date_cols = ['founded_at', 'first_funding_at', 'last_funding_at']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

log_step(3, 'Parse date columns to datetime', before, df.shape,
         f'Columns parsed: {date_cols}')

print('Date column dtypes:')
for col in date_cols:
    print(f'  {col}: {df[col].dtype} | Missing: {df[col].isna().sum():,}')

[Step 3] Parse date columns to datetime | Rows: 54,294 → 54,294 (+0)
Date column dtypes:
  founded_at: datetime64[ns] | Missing: 15,741
  first_funding_at: datetime64[ns] | Missing: 4,866
  last_funding_at: datetime64[ns] | Missing: 4,862


In [10]:
before = df.shape

str_cols = ['name', 'market', 'status', 'country_code', 'state_code', 'region', 'city', 'category_list']
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({'nan': np.nan, 'None': np.nan, '': np.nan})

log_step(4, 'Strip whitespace from string columns', before, df.shape,
         f'Cleaned columns: {str_cols}')

[Step 4] Strip whitespace from string columns | Rows: 54,294 → 54,294 (+0)


In [11]:
before = df.shape

print('Raw status values:')
print(df['status'].value_counts(dropna=False).to_string())

df['status'] = df['status'].str.lower().str.strip()

# Map to standard labels
status_map = {
    'operating': 'operating',
    'closed': 'closed',
    'acquired': 'acquired',
    'ipo': 'ipo'
}
df['status'] = df['status'].map(status_map)

log_step(5, 'Standardise status column to 4 categories', before, df.shape,
         'Values not in [operating, closed, acquired, ipo] → NaN')

print('\nCleaned status distribution:')
print(df['status'].value_counts(dropna=False).to_string())

Raw status values:
status
operating    41829
NaN           6170
acquired      3692
closed        2603
[Step 5] Standardise status column to 4 categories | Rows: 54,294 → 54,294 (+0)

Cleaned status distribution:
status
operating    41829
NaN           6170
acquired      3692
closed        2603


In [12]:
before = df.shape
df = df.dropna(subset=['status'])
log_step(6, 'Drop rows with missing status', before, df.shape,
         'status is the primary analytical target variable; cannot impute')

[Step 6] Drop rows with missing status | Rows: 54,294 → 48,124 (-6,170)


In [13]:
before = df.shape
df = df.drop_duplicates(subset=['name', 'country_code', 'founded_year'], keep='first')
log_step(7, 'Remove duplicate records', before, df.shape,
         'Deduplicated on (name, country_code, founded_year)')

[Step 7] Remove duplicate records | Rows: 48,124 → 48,109 (-15)


In [14]:
before = df.shape

round_cols = ['seed', 'venture', 'equity_crowdfunding', 'undisclosed',
              'convertible_note', 'debt_financing', 'angel', 'grant',
              'private_equity', 'post_ipo_equity', 'post_ipo_debt',
              'secondary_market', 'product_crowdfunding',
              'round_A', 'round_B', 'round_C', 'round_D', 'round_E',
              'round_F', 'round_G', 'round_H']

for col in round_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

log_step(8, 'Convert funding round columns to numeric and fill NaN with 0', before, df.shape,
         f'{len(round_cols)} round columns standardised')

[Step 8] Convert funding round columns to numeric and fill NaN with 0 | Rows: 48,109 → 48,109 (+0)


In [15]:
before = df.shape

# 1. Startup age at first funding (years)
df['days_to_first_funding'] = (df['first_funding_at'] - df['founded_at']).dt.days

# 2. Funding duration (first to last funding in days)
df['funding_duration_days'] = (df['last_funding_at'] - df['first_funding_at']).dt.days

# 3. Average funding per round
df['avg_funding_per_round'] = np.where(
    df['funding_rounds'] > 0,
    df['funding_total_usd'] / df['funding_rounds'],
    0
)

# 4. Is US-based?
df['is_usa'] = (df['country_code'] == 'USA').astype(int)

# 5. Primary category (first pipe-separated value)
df['primary_category'] = df['category_list'].astype(str).str.split('|').str[1].str.strip()
df['primary_category'] = df['primary_category'].replace({'nan': np.nan, 'None': np.nan})

# 6. Binary outcome — did the startup fail (closed)?
df['is_closed'] = (df['status'] == 'closed').astype(int)

# 7. Has reached Series A or beyond?
df['reached_series_a'] = ((df['round_A'] > 0) | (df['round_B'] > 0) | (df['round_C'] > 0)).astype(int)

# 8. Founding decade
df['founding_decade'] = (df['founded_year'] // 10 * 10).astype('Int64')

log_step(9, 'Engineer analytical features', before, df.shape,
         'Added: days_to_first_funding, funding_duration_days, avg_funding_per_round, is_usa, primary_category, is_closed, reached_series_a, founding_decade')

print('\nNew feature summary:')
new_cols = ['days_to_first_funding', 'funding_duration_days', 'avg_funding_per_round', 'is_closed', 'reached_series_a']
print(df[new_cols].describe().round(2).to_string())

[Step 9] Engineer analytical features | Rows: 48,109 → 48,109 (+0)

New feature summary:
       days_to_first_funding  funding_duration_days  avg_funding_per_round  is_closed  reached_series_a
count               37618.00               48099.00           3.978700e+04   48109.00          48109.00
mean                 1408.77                 314.79           8.049256e+06       0.05              0.25
std                  3398.09                 627.45           6.358220e+07       0.23              0.43
min                -17536.00                   0.00           1.000000e+00       0.00              0.00
25%                   151.00                   0.00           2.667083e+05       0.00              0.00
50%                   546.00                   0.00           1.350000e+06       0.00              0.00
75%                  1484.00                 389.00           5.400000e+06       0.00              0.00
max                 83774.00               17287.00           6.015901e+09     

In [16]:
before = df.shape
# Keep startups founded between 1990 and 2023 (reliable era for VC data)
df = df[(df['founded_year'].isna()) | ((df['founded_year'] >= 1990) & (df['founded_year'] <= 2023))]
log_step(10, 'Filter to founded year 1990–2023', before, df.shape,
         'Removed records with implausible founding dates outside VC era')

[Step 10] Filter to founded year 1990–2023 | Rows: 48,109 → 47,179 (-930)


In [17]:
etl_df = pd.DataFrame(etl_log)
print('=== ETL PIPELINE SUMMARY ===')
print(etl_df.to_string(index=False))
print(f'\nInitial rows: {initial_shape[0]:,}')
print(f'Final rows:   {df.shape[0]:,}')
print(f'Total dropped: {initial_shape[0] - df.shape[0]:,} ({(initial_shape[0] - df.shape[0])/initial_shape[0]*100:.1f}%)')

=== ETL PIPELINE SUMMARY ===
 Step                                                  Description  Rows Before  Rows After  Rows Dropped                                                                                                                                             Detail
    1                           Strip whitespace from column names        54294       54294             0                                                                      Columns like " market " and " funding_total_usd " had leading/trailing spaces
    2                         Convert funding_total_usd to numeric        54294       54294             0                                                                                                Stripped commas, dashes, spaces; converted to float
    3                               Parse date columns to datetime        54294       54294             0                                                                              Columns parsed: ['founded_at'

In [18]:
print('Final cleaned dataset info:')
print(f'  Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
print()
print('Status distribution in cleaned data:')
print(df['status'].value_counts().to_string())
print()
print('Missing values in key columns:')
key_cols = ['status', 'funding_total_usd', 'country_code', 'founded_year', 'funding_rounds', 'is_closed']
print(df[key_cols].isnull().sum().to_string())

Final cleaned dataset info:
  Shape: 47,179 rows x 47 columns
  Memory: 46.8 MB

Status distribution in cleaned data:
status
operating    41027
acquired      3571
closed        2581

Missing values in key columns:
status                   0
funding_total_usd     8132
country_code          5029
founded_year         10556
funding_rounds           0
is_closed                0


In [25]:
from google.colab import files

clean_path = '/content/startups_cleaned.csv'
df.to_csv(clean_path, index=False)

files.download(clean_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>